# ☁️ Start here: take the course on Google Colab

**Run this notebook once per Colab account** (and again whenever you want the latest course). It takes 3–5 minutes.

| | |
|---|---|
| ☁️ Google Colab | [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/diego28e/langchain-langgraph-mastery/blob/main/colab/START_HERE.ipynb) |
| ⏱️ Time | 3–5 min (first run), under 1 min after that |
| 🔑 Keys | Optional now: add them later in **Colab Secrets** (🔑 in the left sidebar), Step 4 |
| 🧠 Runtime | The default **Python 3 / CPU** runtime is enough (a GPU is only for Lesson 14.04's Colab notebook) |
| 📄 Guide | [docs/COLAB.md](https://github.com/diego28e/langchain-langgraph-mastery/blob/main/docs/COLAB.md): what runs on Colab, what needs your PC, troubleshooting |

**What it does:**
1. Mounts your **Google Drive** and clones the course into `MyDrive/langchain-langgraph-mastery` (or updates it with `git pull`). Your clone and everything the lessons write (`data/outputs/`) stay in Drive between sessions.
2. Installs the course into this runtime with versions tested on Colab (`pip install -e ".[colab]"` plus `constraints/colab-2026-10.txt`).
3. Loads your **Colab Secrets** (API keys) and shows a ✅/❌ table.

**How:** **Runtime → Run all**. Allow Drive access when Colab asks. If Step 3 stops with **"Restart once"**, do **Runtime → Restart session**, then **Runtime → Run all** again: the second run skips the install and finishes. That's all.

> **Prefer your own PC?** Close this notebook and follow [00-02-create-course-env.md](https://github.com/diego28e/langchain-langgraph-mastery/blob/main/modules/00-foundations/00-02-create-course-env.md). You can mix both: most lessons run on either, a few need your PC ([the list](https://github.com/diego28e/langchain-langgraph-mastery/blob/main/docs/COLAB.md#per-module)).

## Step 1 · Settings

Leave these as they are unless you forked the course to your own GitHub account.

In [ ]:
# Step 1: Settings (edit only for your own fork or another branch).
REPO_URL = "https://github.com/diego28e/langchain-langgraph-mastery.git"
BRANCH = "main"
USE_DRIVE = True  # True: keep the clone in Google Drive (survives sessions). False: /content (deleted when the runtime ends).

## Step 2 · Mount Drive and get the course

The first time, this **clones** the course. Later runs **update** it with `git pull --ff-only`, which never overwrites your work: if you edited course files, it warns and keeps your copy as it is.

In [ ]:
# Step 2: Mount Google Drive, then clone the course (first time) or update it (later runs).
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
REPO_NAME = "langchain-langgraph-mastery"
BASE = Path("/content/drive/MyDrive") if USE_DRIVE else Path("/content")
REPO_ROOT = BASE / REPO_NAME


def git(*args: str, cwd: Path | None = None) -> tuple[int, str]:
    """Run git and return (exit code, its output)."""
    result = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
    return result.returncode, (result.stdout + result.stderr).strip()


if not IN_COLAB:
    print("ℹ️ This notebook sets up Google Colab. On your PC, follow modules/00-foundations/00-02-create-course-env.md.")
else:
    if USE_DRIVE and not BASE.is_dir():
        from google.colab import drive

        drive.mount("/content/drive")  # Colab asks you to allow access.
    if (REPO_ROOT / ".git").is_dir():
        code, out = git("pull", "--ff-only", "origin", BRANCH, cwd=REPO_ROOT)
        if code == 0:
            print("✅ Course updated:", REPO_ROOT, "·", out.splitlines()[-1] if out else "")
        else:
            print("⚠️ Could not update the course, so your clone stays exactly as it is (it still works).")
            print("   Usual cause: you edited course files. Keep them and carry on, or see docs/COLAB.md#update.")
            print("  ", out.replace("\n", "\n   "))
    elif REPO_ROOT.exists():
        raise FileExistsError(f"{REPO_ROOT} exists but is not a git clone. Rename it, then run this cell again.")
    else:
        print("⏳ Cloning the course into", REPO_ROOT, "...")
        code, out = git("clone", "--branch", BRANCH, REPO_URL, str(REPO_ROOT))
        if code != 0:
            raise RuntimeError("git clone failed:\n" + out)
        print("✅ Course cloned:", REPO_ROOT)
    git("config", "core.fileMode", "false", cwd=REPO_ROOT)  # Drive drops file modes; don't show them as changes.

## Step 3 · Install the course into this runtime

Each Colab notebook runs in its **own runtime**, so every lesson's setup cell repeats this step automatically the first time you run it (about 1–3 minutes; after that it is skipped until the runtime is recycled). Here you do it once to check everything works.

If this cell stops with **"Restart once"**: **Runtime → Restart session**, then **Runtime → Run all**.

In [ ]:
# Step 3: Install the course (once per runtime) with the Colab-tested versions, then load Colab Secrets.
if IN_COLAB:
    if str(REPO_ROOT / "src") not in sys.path:
        sys.path.insert(0, str(REPO_ROOT / "src"))
    from mastery.colab import prepare_runtime

    prepare_runtime(REPO_ROOT)  # Raises RestartRequired the first time if Colab had already imported an upgraded package.

## Step 4 · Your API keys: Colab Secrets

Colab keeps keys out of your notebooks: click **🔑 Secrets** in the left sidebar → **Add new secret**, use the **same names as in `.env.example`**, and switch **Notebook access** on. The course reads them by itself (`load_env()` copies them into the environment and never overwrites a variable you set yourself).

| Secret | Needed for |
|---|---|
| `GROQ_API_KEY` | Most 💸 cells (free tier, Lesson 00.04 shows how to get it) |
| `LANGSMITH_API_KEY` | Traces (Lesson 01.08), optional |
| `TAVILY_API_KEY` | Web search lessons, optional |
| `GOOGLE_API_KEY` | Gemini lessons, optional |

Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs. After you add a secret, run the next cell again (no restart needed). [All the names](https://github.com/diego28e/langchain-langgraph-mastery/blob/main/docs/COLAB.md#secrets)

In [ ]:
# Step 4: Which keys does this runtime see? (Names only, never values.)
if IN_COLAB:
    from mastery.config import is_set, load_colab_secrets

    loaded = load_colab_secrets(force=True)  # force: re-read after you add a secret
    print("Loaded from Colab Secrets:", ", ".join(loaded) or "(none yet)")
    for name in ("GROQ_API_KEY", "LANGSMITH_API_KEY", "TAVILY_API_KEY", "GOOGLE_API_KEY"):
        print(f"  {'✅' if is_set(name) else '➖'} {name}")

## Step 5 · Verify

The same idea as Lesson 00.02: which Python and which packages this runtime really uses.

In [ ]:
# Step 5: A verification table for this runtime.
import platform
from importlib import metadata

from IPython.display import Markdown, display


def installed(name: str) -> str:
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return "❌ not installed"


if IN_COLAB:
    import mastery

    in_repo = Path(mastery.__file__).resolve().is_relative_to(REPO_ROOT.resolve())
    rows = [
        ("Python", platform.python_version(), "✅" if sys.version_info >= (3, 11) else "❌ needs 3.11+"),
        ("Course clone", str(REPO_ROOT), "✅ in Drive" if USE_DRIVE else "⚠️ temporary (/content)"),
        ("mastery", f"{mastery.__version__} from {Path(mastery.__file__).parent}", "✅" if in_repo else "❌ restart the session"),
    ]
    for package in ("langchain", "langchain-core", "langgraph", "langchain-groq", "langsmith", "pandas", "pytest"):
        version = installed(package)
        rows.append((package, version, "❌" if version.startswith("❌") else "✅"))
    rows.append(("GROQ_API_KEY", "set" if is_set("GROQ_API_KEY") else "not set", "✅" if is_set("GROQ_API_KEY") else "➖ optional"))
    table = "| Check | Value | |\n|---|---|---|\n" + "\n".join(f"| {a} | `{b}` | {c} |" for a, b, c in rows)
    display(Markdown(table))

**You should now see** ✅ on every row (➖ for keys you haven't added yet). A ❌ on `mastery` means this kernel had already imported another copy: **Runtime → Restart session**, then **Run all**.

---
## ➡️ What's next

1. Open a lesson from GitHub with its **Open in Colab** badge, or from Drive (**File → Open notebook → Google Drive** → `langchain-langgraph-mastery/modules/...`). Opening from Drive saves your edits in your clone.
2. Run it top to bottom. Its first code cell finds this clone, installs the course into that runtime (once) and loads your secrets. If it asks you to **restart once**, do it and run all again.
3. Lessons marked **❌ Not available on Colab** (Docker, Ollama, Studio) need your PC or the devcontainer: [the list and the setup](https://github.com/diego28e/langchain-langgraph-mastery/blob/main/docs/COLAB.md#per-module).

Good next stops: [00.03 Jupyter fundamentals](https://colab.research.google.com/github/diego28e/langchain-langgraph-mastery/blob/main/modules/00-foundations/00-03-jupyter-fundamentals.ipynb) · [00.04 Secrets](https://colab.research.google.com/github/diego28e/langchain-langgraph-mastery/blob/main/modules/00-foundations/00-04-secrets-and-env.ipynb) · [01.01 Chat models](https://colab.research.google.com/github/diego28e/langchain-langgraph-mastery/blob/main/modules/01-models-messages-prompts/01-01-chat-models.ipynb)